# 1. Data Ingestion

In [16]:
import pyodbc
import numpy as np
import pandas as pd

In [17]:
customers = pd.read_csv('olist_data/olist_customers_dataset.csv')
geolocation = pd.read_csv('olist_data/olist_geolocation_dataset.csv')
order_items = pd.read_csv('olist_data/olist_order_items_dataset.csv')
order_payments = pd.read_csv('olist_data/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('olist_data/olist_order_reviews_dataset.csv')
orders_dataset = pd.read_csv('olist_data/olist_orders_dataset.csv')
products = pd.read_csv('olist_data/olist_products_dataset.csv')
sellers = pd.read_csv('olist_data/olist_sellers_dataset.csv')
product_category_name_translation = pd.read_csv('olist_data/product_category_name_translation.csv')

# 2. Check Duplicates& Missing Values

In [18]:
datasets = {
    'customers':customers,
    'geolocation':geolocation,
    'order_items':order_items,
    'order_payments':order_payments,
    'order_reviews':order_reviews,
    'orders_dataset':orders_dataset,
    'products':products,
    'sellers':sellers,
    'product_category_name_translation':product_category_name_translation
}

In [19]:
for name, set in datasets.items():
    duplicate_count = set.duplicated().sum()
    if duplicate_count:
        print (f'--{name.upper()} have {duplicate_count} duplicates.')
        set.drop_duplicates(inplace=True)
        print (f'--Droped {duplicate_count} rows from {name.upper()}.')
    else:
        print (f'-{name.upper()} have no duplicates.')
    print('=' * 49)

-CUSTOMERS have no duplicates.
--GEOLOCATION have 261831 duplicates.
--Droped 261831 rows from GEOLOCATION.
-ORDER_ITEMS have no duplicates.
-ORDER_PAYMENTS have no duplicates.
-ORDER_REVIEWS have no duplicates.
-ORDERS_DATASET have no duplicates.
-PRODUCTS have no duplicates.
-SELLERS have no duplicates.
-PRODUCT_CATEGORY_NAME_TRANSLATION have no duplicates.


In [20]:
for name, set in datasets.items():
    null_rows = set.isna().sum()
    missing_vals = null_rows[null_rows > 0]
    print (f'========== {name.upper()} Missing Values ==========')
    if not missing_vals.empty:
        print(missing_vals)
        print(f'\n-Missing Values Percents: \n{round(missing_vals / set.shape[0] * 100, 2)}')
    else:
        print ('NO Missing Values.')
    print('_' * 49+'\n')


========== CUSTOMERS Missing Values ==========
NO Missing Values.
_________________________________________________

========== GEOLOCATION Missing Values ==========
NO Missing Values.
_________________________________________________

========== ORDER_ITEMS Missing Values ==========
NO Missing Values.
_________________________________________________

========== ORDER_PAYMENTS Missing Values ==========
NO Missing Values.
_________________________________________________

========== ORDER_REVIEWS Missing Values ==========
review_comment_title      87656
review_comment_message    58247
dtype: int64

-Missing Values Percents: 
review_comment_title      88.34
review_comment_message    58.70
dtype: float64
_________________________________________________

========== ORDERS_DATASET Missing Values ==========
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

-Missing Values Percents: 
order_approved_at             

In [21]:
orders_dataset[orders_dataset.order_delivered_customer_date.isna()].groupby('order_status').order_id.count()
# Missing dates align with natural order status so it won't be removed because deleting them would affect the analysis ahead 

order_status
approved          2
canceled        619
created           5
delivered         8
invoiced        314
processing      301
shipped        1107
unavailable     609
Name: order_id, dtype: int64

# 3. Handling Missing values

## 3.1 Products

In [22]:
# Products are linked to real sales and revenue so we keep the null names with changeing their values to 'unknown'
products.fillna({'product_category_name':'unknown'}, inplace = True)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [23]:
# Fill missing numbers (weights, sizes, photo quantities) with 0 to keep data types consistent.
# We can't fill data with mean/ median becasue every product is different and there is too many products for Group-wise (Subgroup) Imputation
empty_product_numeric_cols = ['product_name_lenght',
                                'product_description_lenght',
                                'product_photos_qty',
                                'product_weight_g',
                                'product_length_cm',
                                'product_height_cm',
                                'product_width_cm']
for col in empty_product_numeric_cols:
    products.fillna({col:0}, inplace=True)
products.isna().sum()

product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

## 3.2 Order_Reviews

In [24]:
# We won't drop the nulls in the reviews because we need the product scores we can just fill them
order_reviews.fillna({'review_comment_title':'No Title',
                    'review_comment_message':'No Message'}, inplace=True)
order_reviews.isna().sum()

review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
dtype: int64

# 4. Data Types

In [25]:
for name, set in datasets.items():
    print (f'========== {name.upper()} Data Types ==========')
    print(set.dtypes)
    print('_' * 49+'\n')

========== CUSTOMERS Data Types ==========
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object
_________________________________________________

========== GEOLOCATION Data Types ==========
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object
_________________________________________________

========== ORDER_ITEMS Data Types ==========
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object
_________________________________________________

========== ORDER_PAYMENTS Data Types ==========
order_id                    str
payment_se

## 4.1 Fix The Dates Data types To Enable Time-based Analysis

In [26]:
order_items.shipping_limit_date = pd.to_datetime(order_items.shipping_limit_date)
order_items.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

In [27]:
order_reviews.review_answer_timestamp = pd.to_datetime(order_reviews.review_answer_timestamp)
order_reviews.dtypes

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date                  str
review_answer_timestamp    datetime64[us]
dtype: object

In [28]:
str_to_date = ['order_purchase_timestamp',
                'order_approved_at',
                'order_delivered_carrier_date',
                'order_delivered_customer_date',
                'order_estimated_delivery_date']
for date in str_to_date:
    orders_dataset[date] = pd.to_datetime(orders_dataset[date])
orders_dataset.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

# 5. Assessing the data size

In [29]:
for name, set in datasets.items():
    print (f'========== {name.upper()} Shape ==========')
    print(set.shape)
    print('_' * 40+'\n')

========== CUSTOMERS Shape ==========
(99441, 5)
________________________________________

========== GEOLOCATION Shape ==========
(738332, 5)
________________________________________

========== ORDER_ITEMS Shape ==========
(112650, 7)
________________________________________

========== ORDER_PAYMENTS Shape ==========
(103886, 5)
________________________________________

========== ORDER_REVIEWS Shape ==========
(99224, 7)
________________________________________

========== ORDERS_DATASET Shape ==========
(99441, 8)
________________________________________

========== PRODUCTS Shape ==========
(32951, 9)
________________________________________

========== SELLERS Shape ==========
(3095, 4)
________________________________________

========== PRODUCT_CATEGORY_NAME_TRANSLATION Shape ==========
(71, 2)
________________________________________



## 5.1 Compressing Geolocation data 
Because it is almost 1M rows
it has the coordinates of alot of similar locations
That would be too much to export to SQL

In [30]:
geo_cleaned = geolocation.groupby('geolocation_zip_code_prefix').agg({'geolocation_lat':'mean',
                                                        'geolocation_lng':'mean',
                                                        'geolocation_city':'first',
                                                        'geolocation_state':'first'}).reset_index()

print(f'Compressed geolocation: {len(geolocation)} rows To geo_cleaned: {len(geo_cleaned)}')

geo_cleaned.columns = ['zip_code_prefix', 'lat', 'lng', 'city', 'state']

geo_cleaned.head()

Compressed geolocation: 738332 rows To geo_cleaned: 19015


,zip_code_prefix,lat,lng,city,state
0,1001,-23.550227,-46.634039,sao paulo,SP
1,1002,-23.547657,-46.634991,sao paulo,SP
2,1003,-23.549000,-46.635582,sao paulo,SP
3,1004,-23.549829,-46.634792,sao paulo,SP
4,1005,-23.549547,-46.636406,sao paulo,SP


# 6. Exporting to SQL

In [35]:
SQL_SERVER = r"BLUE\SQLEXPRESS"
SQL_DATABASE = "Olist"

connection_string = (
    r"DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={SQL_SERVER};"
    f"DATABASE={SQL_DATABASE};"
    r"Trusted_Connection=yes;"
    r"Encrypt=yes;"
    r"TrustServerCertificate=yes;"
)
print(connection_string)

DRIVER={ODBC Driver 18 for SQL Server};SERVER=BLUE\SQLEXPRESS;DATABASE=Olist;Trusted_Connection=yes;Encrypt=yes;TrustServerCertificate=yes;


In [ ]:
# ---------------------------------------------------------------------------
# 1. Schema definition: one place that says, per table, which columns are the
#    primary key and which columns are foreign keys pointing where.
#
#    - Order in this dict = table creation order. A table only needs to come
#      after the tables its FKs point to (FKs themselves are added later,
#      after every table exists, so creation order just avoids surprises).
#    - "primary_key" can be a single column or a list (composite key).
#    - "foreign_keys" is a list of dicts: columns -> ref_table(ref_columns).
# ---------------------------------------------------------------------------

TABLE_CONFIG = {
    "product_category_translation": {
        "frame": product_category_name_translation,
        "primary_key": ["product_category_name"],
        "foreign_keys": [],
    },
    "customer": {
        "frame": customers,
        "primary_key": ["customer_id"],
        "foreign_keys": [],
    },
    "seller": {
        "frame": sellers,
        "primary_key": ["seller_id"],
        "foreign_keys": [],
    },
    "geolocation": {
        "frame": geo_cleaned,
        # No natural unique key in the raw Olist data (many rows share a
        # zip prefix). Leave this empty unless your aggregation guarantees
        # uniqueness on geolocation_zip_code_prefix.
        "primary_key": [],
        "foreign_keys": [],
    },
    "product": {
        "frame": products,
        "primary_key": ["product_id"],
        # No FK to product_category_translation: some product_category_name
        # values in the raw data have no matching row in the translation
        # table, which breaks the constraint.
        "foreign_keys": [],
    },
    "order": {
        "frame": orders_dataset,
        "primary_key": ["order_id"],
        "foreign_keys": [
            {
                "columns": ["customer_id"],
                "ref_table": "customer",
                "ref_columns": ["customer_id"],
            },
        ],
    },
    "order_item": {
        "frame": order_items,
        "primary_key": ["order_id", "order_item_id"],
        "foreign_keys": [
            {"columns": ["order_id"], "ref_table": "order", "ref_columns": ["order_id"]},
            {"columns": ["product_id"], "ref_table": "product", "ref_columns": ["product_id"]},
            {"columns": ["seller_id"], "ref_table": "seller", "ref_columns": ["seller_id"]},
        ],
    },
    "order_payment": {
        "frame": order_payments,
        "primary_key": ["order_id", "payment_sequential"],
        "foreign_keys": [
            {"columns": ["order_id"], "ref_table": "order", "ref_columns": ["order_id"]},
        ],
    },
    "order_review": {
        "frame": order_reviews,
        "primary_key": [],
        "foreign_keys": [
            {"columns": ["order_id"], "ref_table": "order", "ref_columns": ["order_id"]},
        ],
    },
}

In [ ]:
# ---------------------------------------------------------------------------
# 2. Helpers
# ---------------------------------------------------------------------------

def sql_type(series):
    dtype = series.dtype
    if pd.api.types.is_datetime64_any_dtype(dtype):
        return "DATETIME2"
    if pd.api.types.is_bool_dtype(dtype):
        return "BIT"
    if pd.api.types.is_integer_dtype(dtype):
        return "BIGINT"
    if pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    return "NVARCHAR(450)" if series.name.endswith("_id") or series.name.endswith("name") else "NVARCHAR(MAX)"
    # NOTE: NVARCHAR(MAX) columns cannot be part of a PK/index in SQL Server.
    # Any column used as a PK or FK must resolve to a sized type, which is
    # why key-like column names above get NVARCHAR(450) instead of MAX.

def sql_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    if isinstance(value, np.generic):
        return value.item()
    return value

def build_column_definitions(frame, pk_columns):
    """Column defs, forcing PK columns to NOT NULL (required by SQL Server)."""
    defs = []
    for column in frame.columns:
        col_type = sql_type(frame[column])
        nullability = "NOT NULL" if column in pk_columns else "NULL"
        defs.append(f"[{column}] {col_type} {nullability}")
    return ", ".join(defs)

def build_primary_key_clause(table_name, pk_columns):
    if not pk_columns:
        return ""
    cols = ", ".join(f"[{c}]" for c in pk_columns)
    return f", CONSTRAINT [PK_{table_name}] PRIMARY KEY ({cols})"

# ---------------------------------------------------------------------------
# 3. Run: create tables (with PKs inline), bulk-load data, then add FKs
# ---------------------------------------------------------------------------

connection = pyodbc.connect(connection_string)
cursor = connection.cursor()
cursor.fast_executemany = True

try:
    # --- Pass 1: drop + create tables with PKs ---
    for table_name, cfg in TABLE_CONFIG.items():
        frame = cfg["frame"]
        pk_columns = cfg["primary_key"]

        column_definitions = build_column_definitions(frame, pk_columns)
        pk_clause = build_primary_key_clause(table_name, pk_columns)

        cursor.execute(
            f"IF OBJECT_ID(N'dbo.{table_name}', N'U') IS NOT NULL "
            f"DROP TABLE dbo.[{table_name}]"
        )
        cursor.execute(
            f"CREATE TABLE dbo.[{table_name}] ({column_definitions}{pk_clause})"
        )
        print(f"Created dbo.{table_name}" + (f" (PK: {pk_columns})" if pk_columns else ""))

    # --- Pass 2: bulk insert data ---
    for table_name, cfg in TABLE_CONFIG.items():
        frame = cfg["frame"]
        columns = list(frame.columns)

        quoted_columns = ", ".join(f"[{c}]" for c in columns)
        placeholders = ", ".join("?" for _ in columns)
        insert_sql = f"INSERT INTO dbo.[{table_name}] ({quoted_columns}) VALUES ({placeholders})"

        batch_size = 5000
        for start in range(0, len(frame), batch_size):
            batch = frame.iloc[start : start + batch_size]
            rows = [
                tuple(sql_value(v) for v in row)
                for row in batch.itertuples(index=False, name=None)
            ]
            cursor.executemany(insert_sql, rows)

        print(f"Loaded dbo.{table_name}: {len(frame):,} rows")

    # --- Pass 3: add foreign keys now that every table + PK exists ---
    # Before each ALTER TABLE, check for orphaned rows (child values with no
    # matching parent). Adding an FK over dirty data raises IntegrityError
    # (23000) and aborts the whole transaction, so we catch that class of
    # problem here and report it clearly instead.
    fk_errors = []

    for table_name, cfg in TABLE_CONFIG.items():
        for i, fk in enumerate(cfg["foreign_keys"], start=1):
            fk_name = f"FK_{table_name}_{fk['ref_table']}_{i}"
            local_cols = fk["columns"]
            ref_table = fk["ref_table"]
            ref_cols = fk["ref_columns"]

            local_cols_sql = ", ".join(f"[{c}]" for c in local_cols)
            ref_cols_sql = ", ".join(f"[{c}]" for c in ref_cols)

            # Multi-column FKs: match on all columns pairwise, any NULL local
            # column is exempt (SQL Server FKs don't enforce on NULLs).
            join_conditions = " AND ".join(
                f"child.[{lc}] = parent.[{rc}]" for lc, rc in zip(local_cols, ref_cols)
            )
            not_null_conditions = " AND ".join(f"child.[{lc}] IS NOT NULL" for lc in local_cols)

            check_sql = f"""
                SELECT TOP 5 {", ".join(f"child.[{c}]" for c in local_cols)}
                FROM dbo.[{table_name}] AS child
                LEFT JOIN dbo.[{ref_table}] AS parent
                    ON {join_conditions}
                WHERE {not_null_conditions} AND parent.[{ref_cols[0]}] IS NULL
            """
            cursor.execute(check_sql)
            violations = cursor.fetchall()

            if violations:
                sample = [tuple(row) for row in violations]
                msg = (
                    f"Skipped {fk_name}: {table_name}({local_cols}) -> "
                    f"{ref_table}({ref_cols}) has orphaned values, e.g. {sample}"
                )
                print(msg)
                fk_errors.append(msg)
                continue

            cursor.execute(
                f"ALTER TABLE dbo.[{table_name}] "
                f"ADD CONSTRAINT [{fk_name}] FOREIGN KEY ({local_cols_sql}) "
                f"REFERENCES dbo.[{ref_table}] ({ref_cols_sql})"
            )
            print(f"Added FK {fk_name}: {table_name}({local_cols}) -> {ref_table}({ref_cols})")

    if fk_errors:
        print(f"\n{len(fk_errors)} foreign key(s) skipped due to data issues — see messages above.")
        print("Fix the source data (or map/drop the orphaned values) and re-run to attach these FKs.")

    connection.commit()
    print("All tables, primary keys, and foreign keys created successfully.")

except Exception:
    connection.rollback()
    raise

finally:
    cursor.close()
    connection.close()

Created dbo.product_category_translation (PK: ['product_category_name'])
Created dbo.customer (PK: ['customer_id'])
Created dbo.seller (PK: ['seller_id'])
Created dbo.geolocation
Created dbo.product (PK: ['product_id'])
Created dbo.order (PK: ['order_id'])
Created dbo.order_item (PK: ['order_id', 'order_item_id'])
Created dbo.order_payment (PK: ['order_id', 'payment_sequential'])
Created dbo.order_review
Loaded dbo.product_category_translation: 71 rows
Loaded dbo.customer: 99,441 rows
Loaded dbo.seller: 3,095 rows
Loaded dbo.geolocation: 19,015 rows
Loaded dbo.product: 32,951 rows
Loaded dbo.order: 99,441 rows
Loaded dbo.order_item: 112,650 rows
Loaded dbo.order_payment: 103,886 rows
Loaded dbo.order_review: 99,224 rows
Added FK FK_order_customer_1: order(['customer_id']) -> customer(['customer_id'])
Added FK FK_order_item_order_1: order_item(['order_id']) -> order(['order_id'])
Added FK FK_order_item_product_2: order_item(['product_id']) -> product(['product_id'])
Added FK FK_order_it